## Implementing Google-Gemmma-3-270M Architecture From Scratch

![img](./imgs/img2.png)

### Get the Libraries

In [1]:
from importlib.metadata import version

pkgs = ["huggingface_hub", "tokenizers", "torch"]

for p in pkgs:
    print(f"{p} version: {version(p)}")

huggingface_hub version: 0.34.4
tokenizers version: 0.21.1
torch version: 2.4.1


- Make a flag which will tell us whether we want to use the BASE_MODEL or the INSTRUCT_MODEL

In [2]:
USE_INSTRUCT_MODEL = True

In [3]:
import torch
import torch.nn as nn 
import torch.nn.functional as F 

### FeedForward

In [4]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(in_features=cfg["emb_dim"], out_features= cfg["hidden_dim"], dtype= cfg["dtype"], bias = False)
        self.fc2 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias = False)
        self.fc3 = nn.Linear(in_features=cfg["hidden_dim"], out_features= cfg["emb_dim"], dtype= cfg["dtype"], bias = False)
        
    def forward(self, x):
        x_fc1 = self.fc1(x)
        x_fc2 = self.fc2(x)
        
        #Apply GELU Activation Function
        x = F.gelu(x_fc1, approximate="tanh") * x_fc2
        return self.fc3(x)

### RMS Normalization

In [5]:
class RMSNorm(nn.Module):
    def __init__(self, emb_dim, eps = 1e-6, bias = False):
        super().__init__()
        self.eps = eps
        
        #TODO: #Gemma3 stores 0-centered weights meaning 
        self.scale = nn.Parameter(torch.zeros(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim)) if bias else None
        
    def forward(self, x):
        input_dtype = x.dtype
        #Convert the input to float
        x_f = x.float()
        #Compute the variance 
        var = x_f.pow(2).mean(dim = -1, keepdim = True)
        x_norm = x_f * torch.rsqrt(var + self.eps)
        out = x_norm * (1.0 + self.scale.float())
        
        if self.shift is not None:
            out = out + self.shift.float()
        
        return out.to(input_dtype)

### RoPE(Rotary Positional Embeddings)

In [6]:
def compute_rope_params(head_dim , theta_base = 10_000, context_length = 4096, dtype = torch.float32):
    assert head_dim % 2 == 0 , "Embedding Dimension should be EVEN"
    
    #Compute the inverse frequencies
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2, dtype = dtype)[: (head_dim // 2)].float() / head_dim))
    
    #Generate Position Indices
    positions = torch.arange(context_length, dtype = dtype)
    
    #Compute the Angles
    angles = positions[:, None] * inv_freq[None, :]   #Shape: (context_len, head_dim // 2)
    
    #Expand angles to match the head_dim
    angles = torch.cat([angles, angles], dim = 1)    #Shape: (context_len, head_dim)
    
    #Precompute Sine & Cosine Values
    cos = torch.cos(angles)
    sin = torch.sin(angles)
    
    return cos, sin

def apply_rope(x, cos, sin):
    batch_size, num_heads, seq_len, head_dim = x.shape
    
    assert head_dim % 2 == 0, "Head Dimension must be even"
    
    #Split x into first half and second half
    x1 = x[..., : head_dim // 2] #First Half
    x2 = x[..., head_dim // 2 : ]  #Second Half
    
    #Adjust Sin & Cos shapes
    cos = cos[:seq_len, :].unsqueeze(0).unsqueeze(0) #Shape: (1, 1, seq_len, head_dim)
    sin = sin[:seq_len, :].unsqueeze(0).unsqueeze(0) #Shape: (1, 1, seq_len, head_dim)
    
    #Apply the rotary transformation
    rotated = torch.cat((-x2, x1), dim = -1)
    x_rotated = (x * cos) + (rotated * sin)
    
    return x_rotated.to(dtype = x.dtype)

### GQA(Grouped Query Attention)

In [7]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_in, num_heads, num_kv_groups, head_dim = None, qk_norm = False, query_pre_attn_scaler = None, dtype = None):
        super().__init__()
        assert num_heads % num_kv_groups == 0, "num_Heads must be divisible by num_kv_groups"
        
        self.num_heads = num_heads
        self.num_kv_groups = num_kv_groups
        self.group_size = num_heads // num_kv_groups
        
        if head_dim is None:
            assert d_in % num_heads == 0, "d_in must be divisible by num_heads if head_dim is not given"
            head_dim = d_in // num_heads
        
        self.head_dim = head_dim
        self.d_out = num_heads * head_dim
        
        #Define Q,K,V proj matrices
        self.W_query = nn.Linear(d_in, self.d_out, bias = False, dtype = dtype)
        self.W_key = nn.Linear(d_in, out_features= self.num_kv_groups * self.head_dim, bias = False, dtype = dtype)
        self.W_value = nn.Linear(d_in, out_features= self.num_kv_groups * self.head_dim, bias = False, dtype = dtype)
        
        self.out_proj = nn.Linear(self.d_out, out_features= d_in, bias = False, dtype = dtype)
        
        #Check if Normalization is to be performed on Query,Key matrices
        if qk_norm:
            self.q_norm = RMSNorm(head_dim, eps = 1e-6)
            self.k_norm = RMSNorm(head_dim, eps = 1e-6)
        else:
            self.q_norm = self.k_norm = None
        
        #Check for Query_Pre_Attention 
        if query_pre_attn_scaler is not None:
            self.scaling = (query_pre_attn_scaler) ** -0.5
        else:
            self.scaling = (head_dim) ** -0.5
        
    
    def forward(self, x, mask, cos, sin):
        b, num_tokens, _ = x.shape
        
        #Apply them PROJECTIONS
        queries = self.W_query(x)   #Shape: (batch, num_tokens, num_heads * head_dim)
        keys = self.W_key(x)        #Shape: (batch, num_tokens, num_kv_groups * head_dim)
        values = self.W_value(x)    #Shape: (batch, num_tokens, num_kv_groups * head_dim)
        
        #Reshape the Q,K,V matrices
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1,2)   #Shape: (b, num_tokens, num_heads, head_dim)------>(b, num_heads, num_tokens, head_dim)
        keys = keys.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)    #Shape: (b, num_tokens, num_kv_groups, head_dim)----->(b, num_kv_groups ,num_tokens, head_dim)
        values = values.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)  ##Shape: (b, num_tokens, num_kv_groups, head_dim)----->(b, num_kv_groups ,num_tokens, head_dim)
        
        #Apply Normalization
        if self.q_norm:
            queries = self.q_norm(queries)
        if self.k_norm:
            keys = self.k_norm(keys)
            
        #Apply RoPE
        queries = apply_rope(queries, cos, sin)
        keys = apply_rope(keys, cos, sin)
        
        #Expand Keys, Values to match number of heads
        keys = keys.repeat_interleave(self.group_size, dim = 1)
        values = values.repeat_interleave(self.group_size, dim = 1)
        
        #Scale Queries{i.e division by (dim) ^ 1/2}
        queries = queries * self.scaling
        
        #ATTENTION IS ALL IT NEEDS
        attn_scores = queries @ keys.transpose(2,3)
        attn_scores = attn_scores.masked_fill(mask, -torch.inf)
        attn_weights = torch.softmax(attn_scores, dim = -1)
        
        context = (attn_weights @ values).transpose(1,2).reshape(b, num_tokens, self.d_out)
        return self.out_proj(context)

### Transformer

In [8]:
class TransformerBlock(nn.Module):
    
    def __init__(self, cfg:dict, attn_type: str):
        super().__init__()
        self.attn_type = attn_type
        
        #ATTENTION IS ALL IT WANTS
        self.att = GroupedQueryAttention(
            d_in=cfg["emb_dim"],
            num_heads = cfg["n_heads"],
            num_kv_groups= cfg["n_kv_groups"],
            head_dim = cfg["head_dim"],
            qk_norm = cfg["qk_norm"],
            query_pre_attn_scaler= cfg["query_pre_attn_scalar"],
            dtype = cfg["dtype"],
        )
        
        self.ff = FeedForward(cfg)
        self.input_layernorm = RMSNorm(cfg["emb_dim"], eps = 1e-6)
        self.post_attention_layernorm = RMSNorm(cfg["emb_dim"], eps = 1e-6)
        self.pre_feedforward_layernorm = RMSNorm(cfg["emb_dim"], eps = 1e-6)
        self.post_feedforward_layernorm = RMSNorm(cfg["emb_dim"], eps = 1e-6)
        
    def forward(self, x, mask_global, mask_local, cos_global, sin_global, cos_local, sin_local):
        
        #Make Shortcut for Residual Connection to Attention Block
        shortcut = x
        x = self.input_layernorm(x)
        
        if self.attn_type == "sliding_attention":
            attn_mask = mask_local
            cos = cos_local
            sin = sin_local
        else:
            attn_mask = mask_global
            cos = cos_global
            sin = sin_global
        
        x_attn = self.att(x, attn_mask, cos, sin)
        x_attn = self.post_attention_layernorm(x_attn)
        x = shortcut + x_attn
            
        #Make shortcut for Residual Connection to FeedForward Block
        shortcut = x 
        x_ffn = self.pre_feedforward_layernorm(x)
        x_ffn = self.ff(x_ffn)
        x_ffn = self.post_feedforward_layernorm(x_ffn)
        x = shortcut + x_ffn
        return x 
        

### Gemma Model

- The Architecture of Gemma-3 Model follows a different type of technique to implement the **attention**; It follows the **Sliding Window** Attention.

![img.png](./imgs/img3.png)

In [9]:
class Gemma3Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg["layer_types"] is not None and len(cfg["layer_types"]) == cfg["n_layers"]
        
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"], dtype = cfg["dtype"])
        self.blocks = nn.ModuleList([
            TransformerBlock(cfg, attn_type) for attn_type in cfg["layer_types"]
        ])
        
        self.final_norm = RMSNorm(cfg["emb_dim"], eps = 1e-6)
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias = False, dtype = cfg["dtype"])
        self.cfg = cfg
        
        #Get the Cos,Sin values for computing the RoPE
        cos_local, sin_local = compute_rope_params(
            head_dim = cfg["head_dim"],
            theta_base = cfg["rope_local_base"],
            context_length = cfg["context_length"],
            dtype = torch.float32,
        )
        
        cos_global, sin_global = compute_rope_params(
            head_dim= cfg["head_dim"],
            theta_base= cfg["rope_base"],
            context_length= cfg["context_length"],
            dtype = torch.float32,
        )
        
        #Make Cos,Sin values as Register_Buffers so that they are moved with the Model but not as learnable parameters.
        self.register_buffer("cos_local", cos_local, persistent=False)
        self.register_buffer("sin_local", sin_local, persistent=False)
        self.register_buffer("sin_global", sin_global, persistent=False)
        self.register_buffer("cos_global", cos_global, persistent=False)
        
    
    def _create_masks(self, seq_len, device):
        ones = torch.ones((seq_len, seq_len), dtype = torch.bool, device = device)
        
        # mask_global
        # [[0 1 1 1 1]
        # [0 0 1 1 1]
        # [0 0 0 1 1]
        # [0 0 0 0 1]
        # [0 0 0 0 0]]
        mask_global = torch.triu(ones, diagonal=1) #It blocks future tokens only, It is a STANDARD-CAUSAL MASK(No future attention)
        # far_past
        # [[0 0 0 0 0]
        # [0 0 0 0 0]
        # [1 0 0 0 0]
        # [1 1 0 0 0]
        # [1 1 1 0 0]]
        far_past = torch.triu(ones, diagonal=self.cfg["sliding_window"]).T #This blocks attention to tokens that are too far back in the past.
        mask_local = mask_global | far_past #It blocks future tokens + too far past tokens; Through this token gives attention to itself and recent past tokens.
        return mask_global, mask_local 
    
    def forward(self, input_ids):
        b, seq_len = input_ids.shape
        x = self.tok_emb(input_ids)*(self.cfg["emb_dim"]**0.5)
        mask_global, mask_local = self._create_masks(seq_len, x.device)
        
        for block in self.blocks:
            x = block(x, mask_global, mask_local, cos_global = self.cos_global, sin_global = self.sin_global, cos_local = self.cos_local, sin_local = self.sin_local)
        
        #Do the Final Normalization
        x = self.final_norm(x)
        logits = self.out_head(x.to(self.cfg["dtype"]))
        return logits

## Initialize Model

In [10]:
GEMMA3_CONFIG_270M = {
    "vocab_size": 262_144,
    "context_length": 32_768,
    "emb_dim": 640,
    "n_heads": 4,
    "n_layers": 18,
    "hidden_dim": 2048,
    "head_dim": 256,
    "qk_norm": True,
    "n_kv_groups": 1,
    "rope_local_base": 10_000.0,
    "rope_base": 1_000_000.0,
    "sliding_window": 512,
      "layer_types": [
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention"
    ],
    "dtype": torch.bfloat16,
    "query_pre_attn_scalar": 256,
}

torch.manual_seed(123)
model = Gemma3Model(GEMMA3_CONFIG_270M)
model

Gemma3Model(
  (tok_emb): Embedding(262144, 640)
  (blocks): ModuleList(
    (0-17): 18 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=640, out_features=1024, bias=False)
        (W_key): Linear(in_features=640, out_features=256, bias=False)
        (W_value): Linear(in_features=640, out_features=256, bias=False)
        (out_proj): Linear(in_features=1024, out_features=640, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=640, out_features=2048, bias=False)
        (fc2): Linear(in_features=640, out_features=2048, bias=False)
        (fc3): Linear(in_features=2048, out_features=640, bias=False)
      )
      (input_layernorm): RMSNorm()
      (post_attention_layernorm): RMSNorm()
      (pre_feedforward_layernorm): RMSNorm()
      (post_feedforward_layernorm): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=640, out_features

- Dummy Forward Pass to check Model Architecture is Correct

In [11]:
x = torch.tensor([1,2,3]).unsqueeze(0)
model(x)

tensor([[[ 0.7500,  0.1060,  0.4844,  ...,  0.9414,  0.3984, -0.2324],
         [-0.3438, -0.0549,  0.8984,  ..., -0.2402,  0.4570,  0.8242],
         [-0.2695, -0.3242,  0.4141,  ...,  0.8711, -0.9648,  0.9883]]],
       dtype=torch.bfloat16, grad_fn=<UnsafeViewBackward0>)

- Print the Number of Parameters of Model

In [12]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")


Total number of parameters: 435,870,336


- Get the Memory Size of our Model

In [13]:
def model_memory_size(model, input_dtype = torch.float32):
    total_params = 0
    total_grads = 0
    for param in model.parameters():
        param_size = param.numel()
        total_params += param_size
        
        if param.requires_grad:
            total_grads += param_size
    
    #Calculate the Buffer Size
    total_buffers = sum(buffer.numel() for buffer in model.buffers())
    
    element_size = torch.tensor(0, dtype = input_dtype).element_size()
    total_memory_bytes = (total_params + total_grads + total_buffers) * element_size
    
    #Convert Bytes to GigaBytes
    total_memory_gb = total_memory_bytes / (1024 ** 3)
    return total_memory_gb

print(f"Memory Size for Float32: {model_memory_size(model):.2f} GB")
print(f"Memory Size for bfloat32: {model_memory_size(model, input_dtype=torch.bfloat16):.2f} GB")

Memory Size for Float32: 3.37 GB
Memory Size for bfloat32: 1.69 GB


In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

## Load Pre-Trained Weights

In [15]:
def load_weights_into_gemma(model, param_config, params):

    def assign(left, right, tensor_name="unknown"):
        if left.shape != right.shape:
            raise ValueError(
                f"Shape mismatch in tensor '{tensor_name}'. Left: {left.shape}, Right: {right.shape}"
            )
        return torch.nn.Parameter(right.clone().detach() if isinstance(right, torch.Tensor) else torch.tensor(right))

    # Embedding weights
    if "model.embed_tokens.weight" in params:
        model.tok_emb.weight = assign(
            model.tok_emb.weight,
            params["model.embed_tokens.weight"],
            "model.embed_tokens.weight",
        )

    # Iterate over transformer layers
    for l in range(param_config["n_layers"]):
        block = model.blocks[l]
        att = block.att
        # Attention projections
        att.W_query.weight = assign(
            att.W_query.weight,
            params[f"model.layers.{l}.self_attn.q_proj.weight"],
            f"model.layers.{l}.self_attn.q_proj.weight",
        )
        att.W_key.weight = assign(
            att.W_key.weight,
            params[f"model.layers.{l}.self_attn.k_proj.weight"],
            f"model.layers.{l}.self_attn.k_proj.weight",
        )
        att.W_value.weight = assign(
            att.W_value.weight,
            params[f"model.layers.{l}.self_attn.v_proj.weight"],
            f"model.layers.{l}.self_attn.v_proj.weight",
        )
        att.out_proj.weight = assign(
            att.out_proj.weight,
            params[f"model.layers.{l}.self_attn.o_proj.weight"],
            f"model.layers.{l}.self_attn.o_proj.weight",
        )
        # QK normalization weights
        att.q_norm.scale = assign(
            att.q_norm.scale,
            params[f"model.layers.{l}.self_attn.q_norm.weight"],
            f"model.layers.{l}.self_attn.q_norm.weight",
        )
        att.k_norm.scale = assign(
            att.k_norm.scale,
            params[f"model.layers.{l}.self_attn.k_norm.weight"],
            f"model.layers.{l}.self_attn.k_norm.weight",
        )
        # Feed forward weights
        block.ff.fc1.weight = assign(
            block.ff.fc1.weight,
            params[f"model.layers.{l}.mlp.gate_proj.weight"],
            f"model.layers.{l}.mlp.gate_proj.weight",
        )
        block.ff.fc2.weight = assign(
            block.ff.fc2.weight,
            params[f"model.layers.{l}.mlp.up_proj.weight"],
            f"model.layers.{l}.mlp.up_proj.weight",
        )
        block.ff.fc3.weight = assign(
            block.ff.fc3.weight,
            params[f"model.layers.{l}.mlp.down_proj.weight"],
            f"model.layers.{l}.mlp.down_proj.weight",
        )
        # LayerNorm weights
        block.input_layernorm.scale = assign(
            block.input_layernorm.scale,
            params[f"model.layers.{l}.input_layernorm.weight"],
            f"model.layers.{l}.input_layernorm.weight",
        )
        block.post_attention_layernorm.scale = assign(
            block.post_attention_layernorm.scale,
            params[f"model.layers.{l}.post_attention_layernorm.weight"],
            f"model.layers.{l}.post_attention_layernorm.weight",
        )
        # Pre‑ and post‑feed forward norms
        pre_key = f"model.layers.{l}.pre_feedforward_layernorm.weight"
        post_key = f"model.layers.{l}.post_feedforward_layernorm.weight"
        if pre_key in params:
            block.pre_feedforward_layernorm.scale = assign(
                block.pre_feedforward_layernorm.scale,
                params[pre_key],
                pre_key,
            )
        if post_key in params:
            block.post_feedforward_layernorm.scale = assign(
                block.post_feedforward_layernorm.scale,
                params[post_key],
                post_key,
            )

    # Final LayerNorm
    if "model.norm.weight" in params:
        model.final_norm.scale = assign(
            model.final_norm.scale,
            params["model.norm.weight"],
            "model.norm.weight",
        )
    # Output head
    if "lm_head.weight" in params:
        model.out_head.weight = assign(
            model.out_head.weight,
            params["lm_head.weight"],
            "lm_head.weight",
        )
    elif "model.embed_tokens.weight" in params:
        # Weight tying: reuse the embedding weights
        model.out_head.weight = assign(
            model.out_head.weight,
            params["model.embed_tokens.weight"],
            "model.embed_tokens.weight",
        )

- login to huggingface to download pre-trained weights

In [18]:
from huggingface_hub import login   
login()

In [19]:
import json
import os
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download, snapshot_download
from pathlib import Path

if USE_INSTRUCT_MODEL:
    repo_id = "google/gemma-3-270m-it"
else:
    repo_id = "google/gemma-3-270m"

local_dir = Path(repo_id).parts[-1]

weights_file = hf_hub_download(repo_id=repo_id, filename = "model.safetensors", local_dir=local_dir)
weight_dict = load_file(weights_file)

load_weights_into_gemma(model, GEMMA3_CONFIG_270M, weight_dict)
model.to(device)


model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

Gemma3Model(
  (tok_emb): Embedding(262144, 640)
  (blocks): ModuleList(
    (0-17): 18 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=640, out_features=1024, bias=False)
        (W_key): Linear(in_features=640, out_features=256, bias=False)
        (W_value): Linear(in_features=640, out_features=256, bias=False)
        (out_proj): Linear(in_features=1024, out_features=640, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=640, out_features=2048, bias=False)
        (fc2): Linear(in_features=640, out_features=2048, bias=False)
        (fc3): Linear(in_features=2048, out_features=640, bias=False)
      )
      (input_layernorm): RMSNorm()
      (post_attention_layernorm): RMSNorm()
      (pre_feedforward_layernorm): RMSNorm()
      (post_feedforward_layernorm): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=640, out_features

## Load Tokenizer

In [21]:
from tokenizers import Tokenizer

class GemmaTokenizer:
    def __init__(self, tokenizer_file_path):
        tok_file = Path(tokenizer_file_path)
        self.tok = Tokenizer.from_file(str(tok_file))
        eos_token = "<end_of_turn>"
        self.pad_token_id = eos_token
        self.eos_token_id = eos_token
    
    def encode(self, text:str)->list[int]:
        return self.tok.encode(text).ids
    
    def decode(self, ids: list[int])->str:
        return self.tok.decode(ids, skip_special_tokens = False)
    

#Function to Define the Chat-Template
def apply_chat_template(user_text):
    return f"<start_of_tur>user\n{user_text}<end_of_turn>\n<start_of_turn>model\n"
    

In [22]:
tokenizer_file_path = os.path.join(local_dir, "tokenizer.json")
if not os.path.exists(tokenizer_file_path):
    try:
        tokenizer_file_path = hf_hub_download(repo_id=repo_id, filename="tokenizer.json", local_dir=local_dir)
    except Exception as e:
        print(f"Failed to download: {e}")
        tokenizer_file_path = "tokenizer.json"

tokenizer = GemmaTokenizer(tokenizer_file_path)

- Tokenizer Test

In [23]:
prompt = "Give me a short introduction to Machine Learning"
prompt = apply_chat_template(prompt)

input_token_ids = tokenizer.encode(prompt)
text = tokenizer.decode(input_token_ids)
text

'<bos><start_of_tur>user\nGive me a short introduction to Machine Learning<end_of_turn>\n<start_of_turn>model\n'

## Model Inference

In [24]:
def generate_text_stream(model, token_ids, max_new_tokens, eos_token_id = None):
    model.eval()
    with torch.no_grad():
        for _ in range(max_new_tokens):
            out = model(token_ids)[:, -1]
            next_token = torch.argmax(out, dim = -1, keepdim = True)
            
            if(eos_token_id is not None and torch.all(next_token == eos_token_id)):
                break
            yield next_token
            
            token_ids = torch.cat([token_ids, next_token], dim = 1)

In [25]:
input_token_ids_tensor = torch.tensor(input_token_ids, device = device).unsqueeze(0)

for token in generate_text_stream(model, input_token_ids_tensor, 400, eos_token_id=tokenizer.encode("<end_of_turn>")[-1]):
    token_id = token.squeeze(0).tolist()
    print(tokenizer.decode(token_id), end = "", flush=True)

Machine Learning is a powerful and versatile field of computer science that enables computers to learn from data without explicit programming. It involves developing algorithms that allow machines to analyze data and make predictions or decisions based on that data. This process can be used for a wide range of applications, from improving products and services to automating tasks and understanding the world around us.
